In [1]:
# Exercise 1: Bank Account

# Part I: BankAccount Class
class BankAccount:
    """
    Represents a basic bank account with a balance, username, password,
    and authentication status.
    """
    def __init__(self, username, password, initial_balance=0):
        """
        Initializes a BankAccount.

        Args:
            username (str): The username for the account.
            password (str): The password for the account.
            initial_balance (int/float): The initial balance of the account.
        """
        if not isinstance(initial_balance, (int, float)) or initial_balance < 0:
            raise ValueError("Initial balance must be a non-negative number.")

        self.username = username
        self.password = password
        self.balance = initial_balance
        self.authenticated = False # Default authentication status

        print(f"Account for {self.username} created with initial balance: {self.balance:.2f}")

    def _check_authenticated(self):
        """Internal method to check if the user is authenticated."""
        if not self.authenticated:
            raise Exception("Authentication required. Please log in first.")

    def authenticate(self, username, password):
        """
        Authenticates the user by matching the provided username and password.

        Args:
            username (str): The username to check.
            password (str): The password to check.

        Returns:
            bool: True if authentication is successful, False otherwise.
        """
        if self.username == username and self.password == password:
            self.authenticated = True
            print(f"Authentication successful for {self.username}.")
            return True
        else:
            print("Authentication failed: Invalid username or password.")
            return False

    def deposit(self, amount):
        """
        Deposits a positive amount to the account balance.
        Requires authentication.

        Args:
            amount (int/float): The amount to deposit.

        Raises:
            Exception: If the amount is not positive or user is not authenticated.
        """
        self._check_authenticated()
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise Exception("Deposit amount must be a positive number.")
        self.balance += amount
        print(f"Deposited {amount:.2f}. New balance: {self.balance:.2f}")

    def withdraw(self, amount):
        """
        Withdraws a positive amount from the account balance.
        Requires authentication.

        Args:
            amount (int/float): The amount to withdraw.

        Raises:
            Exception: If the amount is not positive, user is not authenticated,
                       or insufficient funds.
        """
        self._check_authenticated()
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise Exception("Withdrawal amount must be a positive number.")
        if self.balance < amount:
            raise Exception("Insufficient funds.")
        self.balance -= amount
        print(f"Withdrew {amount:.2f}. New balance: {self.balance:.2f}")

# Part II: MinimumBalanceAccount
class MinimumBalanceAccount(BankAccount):
    """
    Represents a bank account with a minimum balance requirement.
    Inherits from BankAccount.
    """
    def __init__(self, username, password, initial_balance=0, minimum_balance=0):
        """
        Initializes a MinimumBalanceAccount.

        Args:
            username (str): The username for the account.
            password (str): The password for the account.
            initial_balance (int/float): The initial balance of the account.
            minimum_balance (int/float): The minimum balance that must be maintained.
        """
        super().__init__(username, password, initial_balance)
        if not isinstance(minimum_balance, (int, float)) or minimum_balance < 0:
            raise ValueError("Minimum balance must be a non-negative number.")
        self.minimum_balance = minimum_balance
        print(f"Minimum balance set to: {self.minimum_balance:.2f}")

    def withdraw(self, amount):
        """
        Overrides the withdraw method to enforce the minimum balance.
        Requires authentication.

        Args:
            amount (int/float): The amount to withdraw.

        Raises:
            Exception: If the amount is not positive, user is not authenticated,
                       insufficient funds, or withdrawal would go below minimum balance.
        """
        self._check_authenticated()
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise Exception("Withdrawal amount must be a positive number.")

        if self.balance - amount < self.minimum_balance:
            raise Exception(f"Withdrawal denied: Balance cannot fall below minimum balance of {self.minimum_balance:.2f}.")

        super().withdraw(amount) # Call parent's withdraw method

# Part IV: BONUS - ATM Class
class ATM:
    """
    Simulates an ATM interface for managing bank accounts.
    """
    def __init__(self, account_list, try_limit=2):
        """
        Initializes the ATM.

        Args:
            account_list (list): A list of BankAccount or MinimumBalanceAccount instances.
            try_limit (int): The maximum number of login attempts.
        """
        # Validate account_list
        if not isinstance(account_list, list):
            raise TypeError("account_list must be a list.")
        for account in account_list:
            if not isinstance(account, (BankAccount, MinimumBalanceAccount)):
                raise TypeError("All items in account_list must be BankAccount or MinimumBalanceAccount instances.")
        self.account_list = account_list

        # Validate try_limit
        if not isinstance(try_limit, int) or try_limit <= 0:
            print("Invalid try_limit input. Setting try_limit to default of 2.")
            self.try_limit = 2
        else:
            self.try_limit = try_limit

        self.current_tries = 0
        self.logged_in_account = None # To store the currently logged in account

        self.show_main_menu()

    def show_main_menu(self):
        """
        Displays the main ATM menu and handles user selection for login or exit.
        """
        while True:
            print("\n--- ATM Main Menu ---")
            print("1. Log In")
            print("2. Exit")
            choice = input("Enter your choice: ").strip()

            if choice == '1':
                self.current_tries = 0 # Reset tries for new login attempt
                while self.current_tries < self.try_limit and self.logged_in_account is None:
                    username = input("Enter username: ").strip()
                    password = input("Enter password: ").strip()
                    self.log_in(username, password)
                    if self.logged_in_account is None: # If login failed
                        self.current_tries += 1
                        if self.current_tries < self.try_limit:
                            print(f"Login failed. {self.try_limit - self.current_tries} tries left.")
                        else:
                            print("Maximum login attempts reached. Shutting down.")
                            return # Exit the main menu loop and program
                if self.logged_in_account: # If successfully logged in, break from inner loop
                    break # Exit main menu loop to proceed to account menu
            elif choice == '2':
                print("Thank you for using the ATM. Goodbye!")
                return # Exit the main menu loop and program
            else:
                print("Invalid choice. Please enter 1 or 2.")

    def log_in(self, username, password):
        """
        Attempts to log in a user. If successful, sets the authenticated status
        of the account and calls show_account_menu.

        Args:
            username (str): The username provided by the user.
            password (str): The password provided by the user.
        """
        self.logged_in_account = None # Reset for new login attempt
        for account in self.account_list:
            # Call the authenticate method of the BankAccount/MinimumBalanceAccount instance
            if account.authenticate(username, password):
                self.logged_in_account = account
                self.show_account_menu(account)
                return
        # If loop finishes and no account matched
        # Error message and try increment handled in show_main_menu loop

    def show_account_menu(self, account):
        """
        Displays the account menu for a logged-in user, allowing deposit, withdraw, or exit.

        Args:
            account (BankAccount or MinimumBalanceAccount): The authenticated account instance.
        """
        while True:
            print(f"\n--- Welcome {account.username}! Account Balance: {account.balance:.2f} ---")
            print("1. Deposit")
            print("2. Withdraw")
            print("3. Log Out")
            account_choice = input("Enter your choice: ").strip()

            try:
                if account_choice == '1':
                    amount = float(input("Enter amount to deposit: "))
                    account.deposit(amount)
                elif account_choice == '2':
                    amount = float(input("Enter amount to withdraw: "))
                    account.withdraw(amount)
                elif account_choice == '3':
                    account.authenticated = False # Log out the account
                    self.logged_in_account = None
                    print(f"{account.username} logged out.")
                    break # Exit account menu
                else:
                    print("Invalid choice. Please enter 1, 2, or 3.")
            except ValueError:
                print("Invalid amount. Please enter a number.")
            except Exception as e:
                print(f"Error: {e}")

# --- Test Cases ---
if __name__ == "__main__":
    print("--- Running Bank Account Exercise Test Cases ---")

    # Part I & III Test: BankAccount
    print("\n--- Testing BankAccount ---")
    try:
        my_bank_account = BankAccount("alice123", "pass123", 100)
        # Test unauthenticated access
        try:
            my_bank_account.deposit(50)
        except Exception as e:
            print(f"Expected error (unauthenticated): {e}")

        my_bank_account.authenticate("alice123", "pass123")
        my_bank_account.deposit(50)
        my_bank_account.withdraw(20)
        try:
            my_bank_account.withdraw(200) # Insufficient funds
        except Exception as e:
            print(f"Expected error (insufficient funds): {e}")
        try:
            my_bank_account.deposit(-10) # Non-positive deposit
        except Exception as e:
            print(f"Expected error (non-positive deposit): {e}")
        print(f"Final balance for {my_bank_account.username}: {my_bank_account.balance:.2f}")

    except Exception as e:
        print(f"Error during BankAccount test: {e}")

    # Part II Test: MinimumBalanceAccount
    print("\n--- Testing MinimumBalanceAccount ---")
    try:
        min_bal_account = MinimumBalanceAccount("bob456", "securepwd", 500, 100)
        min_bal_account.authenticate("bob456", "securepwd")
        min_bal_account.deposit(50)
        min_bal_account.withdraw(300) # Balance becomes 250 (above 100)
        print(f"Balance after withdraw 300: {min_bal_account.balance:.2f}")
        try:
            min_bal_account.withdraw(160) # Balance would become 90 (below 100)
        except Exception as e:
            print(f"Expected error (below minimum balance): {e}")
        print(f"Final balance for {min_bal_account.username}: {min_bal_account.balance:.2f}")

    except Exception as e:
        print(f"Error during MinimumBalanceAccount test: {e}")

    # Part IV Test: ATM Class
    print("\n--- Testing ATM ---")
    # Create some accounts for the ATM
    atm_account1 = BankAccount("user1", "pass1", 200)
    atm_account2 = MinimumBalanceAccount("user2", "pass2", 1000, 200)
    atm_account3 = BankAccount("admin", "adminpass", 5000)

    accounts_for_atm = [atm_account1, atm_account2, atm_account3]

    # Test ATM with valid accounts and default try_limit
    print("\n--- ATM Simulation 1 (Normal Login) ---")
    atm = ATM(accounts_for_atm)

    # Test ATM with invalid try_limit
    print("\n--- ATM Simulation 2 (Invalid try_limit) ---")
    try:
        atm_invalid_limit = ATM(accounts_for_atm, try_limit=0)
    except Exception as e:
        print(f"Error initializing ATM with invalid limit. Correctly handled: {e}") # This won't be hit due to internal handling

    # Test ATM with max tries reached (uncomment to test this flow)
    # print("\n--- ATM Simulation 3 (Max Tries Reached) ---")
    # atm_max_tries = ATM(accounts_for_atm, try_limit=2) # This will prompt for login twice
    # For testing, enter wrong credentials twice.


--- Running Bank Account Exercise Test Cases ---

--- Testing BankAccount ---
Account for alice123 created with initial balance: 100.00
Expected error (unauthenticated): Authentication required. Please log in first.
Authentication successful for alice123.
Deposited 50.00. New balance: 150.00
Withdrew 20.00. New balance: 130.00
Expected error (insufficient funds): Insufficient funds.
Expected error (non-positive deposit): Deposit amount must be a positive number.
Final balance for alice123: 130.00

--- Testing MinimumBalanceAccount ---
Account for bob456 created with initial balance: 500.00
Minimum balance set to: 100.00
Authentication successful for bob456.
Deposited 50.00. New balance: 550.00
Withdrew 300.00. New balance: 250.00
Balance after withdraw 300: 250.00
Expected error (below minimum balance): Withdrawal denied: Balance cannot fall below minimum balance of 100.00.
Final balance for bob456: 250.00

--- Testing ATM ---
Account for user1 created with initial balance: 200.00
Acco